# 12 — What's new in ConnInfPy 2.0

Self-contained walkthrough of the core ConnInfPy 2.0 workflow features on synthetic data:

1. `compute_connectivity()` from BOLD time series (Pearson & Ledoit–Wolf partial correlation + `n_valid_tr`).
2. `analyze()` with `two_tailed=True`, `Group × Condition` / `interactions`, and pre-inference `motion_qc`.
3. Unit-preserving summaries: `component_table()` (NBS), `node_summary()`, `network_summary()`, and `coupling()` + `plot_coupling_classes()`.
4. Opt-in BNA-246 + SUIT cerebellar annotation (`cerebellum='anatom'`).


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import conninfpy
from conninfpy import AtlasInfo, BrainnetomeReference, analyze, compute_connectivity
from conninfpy.interpret import annotate_rois
from conninfpy.plot import plot_coupling_classes

rng = np.random.default_rng(42)
print('ConnInfPy version:', conninfpy.__version__)


## 1. Time series → connectivity matrices (`compute_connectivity`)

In [ ]:
# Simulate 24 subjects x 80 timepoints x 16 ROIs with scrubbing NaNs
ts_list = []
for s in range(24):
    ts = rng.normal(0, 1, size=(80, 16))
    if s < 12:  # plant positive coupling between ROIs 0-3
        shared = rng.normal(0, 0.8, size=(80, 1))
        ts[:, :4] += shared
    if s % 5 == 0:  # simulate scrubbed frames
        ts[:10, :] = np.nan
    ts_list.append(ts)

r_pearson, diag_p = compute_connectivity(ts_list, kind='correlation', return_diagnostics=True)
r_lw_part, diag_lw = compute_connectivity(ts_list, kind='partial_correlation', shrinkage='ledoit_wolf', return_diagnostics=True)
print('Pearson shape:', r_pearson.shape, '| n_valid_tr min/max:', min(diag_p['n_valid_tr']), max(diag_p['n_valid_tr']))
print('LW partial shrinkage median:', float(np.median(diag_lw['delta_s'])))


## 2. `analyze()` with `two_tailed=True`, `interactions`, and `motion_qc`

In [ ]:
age = np.linspace(20.0, 50.0, 24)
group = np.array([1.0] * 12 + [0.0] * 12)
fd = rng.uniform(0.08, 0.30, size=24)
atlas = AtlasInfo(
    labels=[f'ROI_{i:02d}' for i in range(16)],
    networks=['Default'] * 4 + ['Visual'] * 4 + ['Somatomotor'] * 4 + ['Cerebellum'] * 4,
)

res_multi = analyze(
    r_pearson,
    input='r',
    interest={'group': group, 'age': age},
    interactions=[('group', 'age')],
    confounds=fd,
    method='tfnbs',       # exact integration (n=None), e=0.4, h=3.0
    n_permutations=100,
    two_tailed=True,
    motion_qc={'fd': fd, 'fd_units': 'mm'},
    atlas=atlas,
    use_mp=False,
    rng=0,
)
print('Contrasts:', list(res_multi.keys()))
res_grp = res_multi['group']
print('Motion QC status:', res_grp.motion_qc['status'])
print('Node summary top 5:\n', res_grp.node_summary(atlas, top_k=5)[['roi_name', 'network', 'n_sig_total', 'n_networks_reached', 'participation']])
print('Network summary top 3:\n', res_grp.network_summary(atlas).head(3)[['network_pair', 'n_sig_positive', 'n_sig_negative']])


## 3. NBS `component_table()`, `coupling()`, and `plot_coupling_classes()`

In [ ]:
res_nbs = analyze(
    group1=r_pearson[12:], group2=r_pearson[:12],
    input='r', test_type='two-sample', method='nbs',
    threshold=2.0, n_permutations=100, use_mp=False, rng=0,
)
print('NBS component table:\n', res_nbs.component_table(atlas))

c_info = res_nbs.inference.coupling()
print('Coupling comparison profiles:', c_info['profiles'])
fig, ax = plt.subplots(figsize=(5.5, 4.8))
plot_coupling_classes(res_nbs.inference, atlas=atlas, alpha=0.05, ax=ax)
plt.close(fig)


## 4. Opt-in BNA-246 + SUIT cerebellar annotation (`cerebellum='anatom'`)

In [ ]:
# Mix cortical and cerebellar MNI coordinates
coords = np.array([
    [-42.0, 20.0, 28.0], [42.0, 20.0, 28.0],
    [-24.0, -64.0, -34.0], [24.0, -64.0, -34.0],  # cerebellum (Crus I / Lobule VI)
])
ref = BrainnetomeReference.load(profiles=False, resolution_mm=1, cerebellum='anatom')
rois = annotate_rois(coords, reference=ref, names=['MFG_L', 'MFG_R', 'Cb_L', 'Cb_R'], support='point')
print(rois.table[['roi', 'lobe', 'gyrus', 'description', 'hemisphere']])
